# SEER Breast Cancer 2010–2017 — Evaluation (04)

Evaluates the predictions saved by `03_modeling.ipynb`, following `Documents/Methods_Modeling.md`, Sections 3.8–6. Nothing is refitted, so this notebook re-runs in minutes.

1. **Threshold selection (Section 3.8).** On inner-val, γ\* = minimum FPR subject to sensitivity ≥ 0.80. TPR and FPR use each cell's own censoring rule and weights; the reference models use IPCW. γ\* is then frozen and applied to the test fold.
2. **Test-fold metrics (Section 4).** All models are evaluated identically, IPCW-weighted against F₁(τ | X):
   - Brier score (primary), AUC;
   - sensitivity, specificity, FPR, precision, F1 and weighted false-positive / false-negative counts at γ\*;
   - calibration-in-the-large (Aalen–Johansen observed risk / mean predicted risk).
3. **Statistics (Section 5).** Mean ± SD over splits, plus factorial contrasts with the corrected repeated-CV t-test.
4. **Tables T2–T8 and S2, and figures** (calibration, decision curves, divergence across horizons), saved to `outputs_evaluation/`.

The notebook works on whatever splits exist in `outputs_modeling/`. With fewer than 2 splits, standard deviations and tests are not available.

In [ ]:
from pathlib import Path
import config
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import seer_modeling as sm

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

NOTEBOOK_DIR = config.WORK_DIR    # all outputs below this folder (config.py)
MODEL_DIR = NOTEBOOK_DIR / "outputs_modeling"
EVAL_DIR = NOTEBOOK_DIR / "outputs_evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

data = sm.load_prep(NOTEBOOK_DIR / "outputs_model_prep")
SPLITS = sorted(
    (int(m[1]), int(m[2]))
    for m in (re.match(r"split_r(\d+)_k(\d+)\.parquet", p.name) for p in MODEL_DIR.glob("split_r*_k*.parquet"))
    if m
)
N_SPLITS = len(SPLITS)
print(f"Splits available: {N_SPLITS} of 15 -> {SPLITS}")

CELL_ORDER = [sm.cell_name(rule, smote) for rule, smote in sm.CELLS]
MODEL_ORDER = list(sm.CLASSIFIERS) + list(sm.REFERENCES)
LABELS = {"lr": "Logistic regression", "rf": "Random forest", "xgb": "XGBoost",
          "cs_cox": "Cause-specific Cox", "cs_rsf": "Cause-specific RSF"}

## 1–2. Threshold selection and test-fold metrics

In [ ]:
metrics = pd.concat(
    [sm.evaluate_split(sm.load_split(MODEL_DIR, r, k), data, r, k) for r, k in SPLITS], ignore_index=True
)
metrics.to_csv(EVAL_DIR / "metrics_by_split.csv", index=False)
print(f"{len(metrics):,} rows = {N_SPLITS} split(s) x {metrics.groupby(['tau', 'model', 'cell']).ngroups} model-cell-horizon combinations")

## Tables T3–T5 — main results by horizon

Mean ± SD over splits. Lower is better for the Brier score; an O/E ratio of 1 means perfect calibration-in-the-large.

In [ ]:
MAIN_METRICS = ["brier", "auc", "oe_ratio", "sensitivity", "specificity", "precision", "f1"]


def mean_sd(s: pd.Series, digits: int = 4) -> str:
    return f"{s.mean():.{digits}f}" if s.size < 2 else f"{s.mean():.{digits}f} ± {s.std(ddof=1):.{digits}f}"


main = (
    metrics.groupby(["tau", "model", "cell"], sort=False)[MAIN_METRICS]
    .agg(mean_sd)
    .reset_index()
)
main["model"] = pd.Categorical(main["model"], MODEL_ORDER)
main["cell"] = pd.Categorical(main["cell"], CELL_ORDER + ["-"])
main = main.sort_values(["tau", "model", "cell"])
for i, tau in enumerate(sm.HORIZONS, start=3):
    table = main[main["tau"] == tau].drop(columns="tau")
    table.to_csv(EVAL_DIR / f"table_T{i}_results_{tau}m.csv", index=False)
    print(f"Table T{i} — {tau} months")
    display(table.set_index(["model", "cell"]))

## Table T7 — factorial decomposition

Paired contrasts within each split, with corrected 95% CIs (Methods Section 5.2):
- **SMOTE effect** within each censoring rule, and its mean (the main effect);
- **censoring contrasts** without SMOTE: IPCW − Zero, IPCW − Discard, Discard − Zero;
- **interaction (H1):** SMOTE effect under Zero − SMOTE effect under IPCW.

For the Brier score, negative values mean an improvement.

In [ ]:
CONTRAST_METRICS = ["brier", "auc", "oe_ratio", "f1", "fpr"]


def contrasts_for(df: pd.DataFrame) -> dict:
    """Per-split contrasts for one (tau, classifier, metric); df has columns split, cell, value."""
    wide = df.pivot(index="split", columns="cell", values="value")
    smote = {rule: wide[f"{rule}+smote"] - wide[rule] for rule in sm.CENSORING_RULES}
    return {
        "SMOTE effect | Zero": smote["zero"],
        "SMOTE effect | Discard": smote["discard"],
        "SMOTE effect | IPCW": smote["ipcw"],
        "SMOTE main effect": (smote["zero"] + smote["discard"] + smote["ipcw"]) / 3,
        "IPCW - Zero": wide["ipcw"] - wide["zero"],
        "IPCW - Discard (weighting)": wide["ipcw"] - wide["discard"],
        "Discard - Zero (labeling)": wide["discard"] - wide["zero"],
        "Interaction (H1)": smote["zero"] - smote["ipcw"],
    }


grid = metrics[metrics["model"].isin(sm.CLASSIFIERS)].assign(split=lambda d: list(zip(d["repeat"], d["fold"])))
rows = []
for (tau, model), g in grid.groupby(["tau", "model"]):
    for metric in CONTRAST_METRICS:
        for name, diffs in contrasts_for(g.rename(columns={metric: "value"})[["split", "cell", "value"]]).items():
            res = sm.corrected_ttest(diffs.to_numpy()) if N_SPLITS >= 2 else {"mean": float(diffs.mean())}
            rows.append({"tau": tau, "model": model, "metric": metric, "contrast": name, **res})
table_t7 = pd.DataFrame(rows)
table_t7.to_csv(EVAL_DIR / "table_T7_factorial.csv", index=False)
display(table_t7[table_t7["metric"] == "brier"].set_index(["tau", "model", "contrast"]).drop(columns="metric").round(5))

## Table T6 — reference models vs the IPCW cell of the same family

Pairs: logistic regression vs cause-specific Cox; random forest vs cause-specific RSF. The comparator is fixed in advance (the IPCW cell, the approach the paper recommends), not selected on test results. Differences are reference − IPCW cell, with corrected 95% CIs.

In [ ]:
PAIRS = {"cs_cox": "lr", "cs_rsf": "rf"}
COMPARATOR_CELL = "ipcw"   # prespecified comparator, not selected on test results
rows = []
for tau in sm.HORIZONS:
    m_tau = metrics[metrics["tau"] == tau].assign(split=lambda d: list(zip(d["repeat"], d["fold"])))
    for ref, clf in PAIRS.items():
        ref_m = m_tau[m_tau["model"] == ref].set_index("split")
        cell_m = m_tau[(m_tau["model"] == clf) & (m_tau["cell"] == COMPARATOR_CELL)].set_index("split")
        for metric in ["brier", "auc", "oe_ratio"]:
            diffs = (ref_m[metric] - cell_m[metric]).to_numpy()
            res = sm.corrected_ttest(diffs) if N_SPLITS >= 2 else {"mean": float(diffs.mean())}
            rows.append({"tau": tau, "reference": ref, "comparator": f"{clf} | {COMPARATOR_CELL}", "metric": metric,
                         "reference_mean": ref_m[metric].mean(), "comparator_mean": cell_m[metric].mean(),
                         **{f"diff_{k}": v for k, v in res.items()}})
table_t6 = pd.DataFrame(rows)
table_t6.to_csv(EVAL_DIR / "table_T6_references.csv", index=False)
display(table_t6.round(4))

## Table T8 — threshold protocol (requested sensitivity 0.80)

In [ ]:
table_t8 = (
    metrics.groupby(["tau", "model", "cell"], sort=False)
    .agg(achieved_sensitivity=("sensitivity", mean_sd), achieved_fpr=("fpr", mean_sd), gamma=("gamma", mean_sd))
    .reset_index()
)
table_t8.insert(3, "requested_sensitivity", sm.TPR_TARGET)
table_t8.to_csv(EVAL_DIR / "table_T8_threshold.csv", index=False)
display(table_t8.set_index(["tau", "model", "cell"]))

## Tables T2 and S2 — training sets per cell, and IPCW weights

In [ ]:
diags = [json.loads((MODEL_DIR / f"split_r{r}_k{k}.json").read_text(encoding="utf-8")) for r, k in SPLITS]
cells = pd.DataFrame([{"column": col, **info} for d in diags for col, info in d["cells"].items()])
cells[["model", "cell", "tau"]] = cells["column"].str.split("|", expand=True)
table_t2 = (
    cells[cells["model"] == "lr"]           # training sets do not depend on the classifier
    .groupby(["tau", "cell"], sort=False)[["n_rows", "n_pos", "n_neg", "ir_before", "ir_after", "n_synthetic", "kish_ess"]]
    .mean().round(2).reset_index()
)
table_t2.to_csv(EVAL_DIR / "table_T2_cells.csv", index=False)
display(table_t2.set_index(["tau", "cell"]))

table_s2 = pd.DataFrame([
    {"tau": int(tau), **{k: v for k, v in w.items() if k != "quantiles_determinable"}, **w["quantiles_determinable"]}
    for d in diags for tau, w in d["weights"].items()
]).groupby("tau").mean().round(3)
table_s2.to_csv(EVAL_DIR / "table_S2_weights.csv")
display(table_s2)

## Figures

Test predictions are pooled within each repeat, i.e. out-of-fold over its 5 folds, and curves are averaged across repeats:
- **Calibration:** deciles of predicted risk vs Aalen–Johansen observed risk.
- **Decision curves:** IPCW-weighted net benefit; the "treat all" line uses the Aalen–Johansen cumulative incidence.
- **Divergence across horizons:** mean Brier score by horizon for each cell.

In [ ]:
def pooled_test(repeat: int) -> pd.DataFrame:
    frames = [sm.load_split(MODEL_DIR, r, k) for r, k in SPLITS if r == repeat]
    df = pd.concat(frames, ignore_index=True)
    return df[df["role"] == "test"].reset_index(drop=True)


REPEATS = sorted({r for r, _ in SPLITS})
POOLED = {r: pooled_test(r) for r in REPEATS}
V_ALL, S_ALL = data.cohort["V"].to_numpy(), data.cohort["status"].to_numpy()


def calibration_points(df: pd.DataFrame, col: str, tau: int, bins: int = 10) -> pd.DataFrame:
    p = df[col].to_numpy(float)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))
    which = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    rows = df["row"].to_numpy()
    return pd.DataFrame([
        {"pred": p[which == b].mean(), "obs": sm.aalen_johansen_cif1(V_ALL[rows[which == b]], S_ALL[rows[which == b]], tau)}
        for b in range(len(edges) - 1) if (which == b).any()
    ])


# ---- Figure presentation: manuscript names, colorblind-checked palette, PLOS ONE size (7.5 in wide, 300 dpi).
# Color = censoring strategy (or reference model); dashed line with open markers = SMOTE-NC.
from matplotlib.lines import Line2D
from PIL import Image

INK, INK2, GRID, AXIS = "#0b0b0b", "#52514e", "#e6e5e0", "#9a9993"
COLOR = {"zero": "#2a78d6", "discard": "#eb6834", "ipcw": "#1baf7a", "cs_cox": "#4a3aa7", "cs_rsf": "#008300"}
STYLE = {c: ls for c, ls in zip(CELL_ORDER, ["-", "-", "-", "--", "--", "--"])}   # also used by later figures
NAME = {"zero": "None (Zero)", "discard": "Discard", "ipcw": "IPCW",
        "zero+smote": "SMOTE-NC (Zero + SMOTE-NC)", "discard+smote": "Discard + SMOTE-NC",
        "ipcw+smote": "IPCW + SMOTE-NC", "cs_cox": "Cause-specific Cox model",
        "cs_rsf": "Cause-specific random survival forest"}
PANELS = [("lr", "Logistic regression"), ("rf", "Random forest"), ("xgb", "XGBoost"),
          ("ref", "Reference survival models")]
PLOS_RC = {"font.family": "Arial", "font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
           "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "axes.edgecolor": AXIS, "xtick.color": INK2,
           "ytick.color": INK2, "axes.labelcolor": INK, "text.color": INK}
PLOS_DIR = EVAL_DIR / "plos_figures"
PLOS_DIR.mkdir(exist_ok=True)


def panel_keys(panel):
    return ["cs_cox", "cs_rsf"] if panel == "ref" else CELL_ORDER


def col_for(panel, key, tau):
    return sm.pred_column(key, "-", tau) if panel == "ref" else sm.pred_column(panel, key, tau)


def line_style(key, markers=True):
    base, smote = key.split("+")[0], key.endswith("+smote")
    return dict(color=COLOR[base], ls="--" if smote else "-", lw=1.3,
                marker="o" if markers else None, ms=3.2, mfc="white" if smote else COLOR[base],
                mec=COLOR[base], mew=0.9)


def frame(axes, xlabel, ylabel):
    for ax, (_, title), letter in zip(axes.flat, PANELS, "ABCD"):
        ax.set_title(f"{letter}   {title}", loc="left", fontweight="bold", color=INK)
        ax.grid(color=GRID, lw=0.5)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[1, :]:
        ax.set_xlabel(xlabel)
    for ax in axes[:, 0]:
        ax.set_ylabel(ylabel)


def shared_legend(fig, extra, markers=True):
    """Legend columns fill top-down; this order reads row-wise: no oversampling / SMOTE-NC / references."""
    def h(k):
        return Line2D([], [], **line_style(k, markers), label=NAME[k])
    col1 = [h("zero"), h("zero+smote"), h("cs_cox")] + extra[:-1]
    col2 = [h("discard"), h("discard+smote"), h("cs_rsf")]
    col3 = [h("ipcw"), h("ipcw+smote"), extra[-1]]
    fig.legend(handles=col1 + col2 + col3, loc="lower center", ncol=3, frameon=False, fontsize=7.5,
               handlelength=2.6, columnspacing=1.6, bbox_to_anchor=(0.5, 0.0))


def save(fig, name, tif=None):
    fig.savefig(EVAL_DIR / name, dpi=300, facecolor="white")
    if tif:  # PLOS ONE: TIFF, RGB without alpha channel, LZW
        fig.savefig(PLOS_DIR / tif, dpi=300, facecolor="white", pil_kwargs={"compression": "tiff_lzw"})
        Image.open(PLOS_DIR / tif).convert("RGB").save(PLOS_DIR / tif, compression="tiff_lzw", dpi=(300, 300))


for tau in sm.HORIZONS:
    with plt.rc_context(PLOS_RC):
        fig, axes = plt.subplots(2, 2, figsize=(7.5, 6.9), sharex=True, sharey=True)
        top = 0
        for ax, (panel, _) in zip(axes.flat, PANELS):
            for key in panel_keys(panel):
                col = col_for(panel, key, tau)
                pts = pd.concat([calibration_points(POOLED[r], col, tau) for r in REPEATS]).groupby(level=0).mean()
                ax.plot(pts["pred"], pts["obs"], **line_style(key))
                top = max(top, pts["pred"].max(), pts["obs"].max())
        lim = np.ceil(top * 10) / 10
        for ax in axes.flat:
            ax.plot([0, lim], [0, lim], color=INK2, lw=0.8, ls=":", zorder=0)
            ax.set_xlim(0, lim)
            ax.set_ylim(0, lim)
        frame(axes, "Mean predicted risk", "Observed risk (Aalen–Johansen)")
        shared_legend(fig, [Line2D([], [], color=INK2, lw=0.8, ls=":", label="Perfect calibration")])
        fig.tight_layout(rect=(0, 0.10, 1, 1), h_pad=1.2, w_pad=1.0)
        save(fig, f"figure_calibration_{tau}m.png", tif="Fig3.tif" if tau == 120 else None)
        plt.show()

In [ ]:
PT = np.round(np.arange(0.01, 0.51, 0.01), 2)
for tau in sm.HORIZONS:
    nb_all = []
    for r in REPEATS:
        rows = POOLED[r]["row"].to_numpy()
        cif = sm.aalen_johansen_cif1(V_ALL[rows], S_ALL[rows], tau)
        nb_all.append(cif - (1 - cif) * PT / (1 - PT))
    treat_all = np.mean(nb_all, axis=0)
    with plt.rc_context(PLOS_RC):
        fig, axes = plt.subplots(2, 2, figsize=(7.5, 6.9), sharex=True, sharey=True)
        for ax, (panel, _) in zip(axes.flat, PANELS):
            for key in panel_keys(panel):
                col = col_for(panel, key, tau)
                curves = []
                for r in REPEATS:
                    df = POOLED[r]
                    y = data.labels[f"y_{tau}"].to_numpy()[df["row"].to_numpy()]
                    det = y >= 0
                    curves.append(sm.net_benefit(df[col].to_numpy(float)[det], (y[det] == 1).astype(int),
                                                 df[f"w_{tau}"].to_numpy(float)[det], len(df), PT))
                ax.plot(PT, np.mean(curves, axis=0), **line_style(key, markers=False))
            ax.plot(PT, treat_all, color=INK2, lw=0.9, ls="-.", zorder=1)
            ax.axhline(0, color=AXIS, lw=0.9, zorder=1)
            ax.set_xlim(0, 0.5)
            ax.set_ylim(-0.01, max(0.02, treat_all.max() * 1.15))
        frame(axes, "Threshold probability", "Net benefit")
        shared_legend(fig, [Line2D([], [], color=INK2, lw=0.9, ls="-.", label="Treat all"),
                            Line2D([], [], color=AXIS, lw=0.9, label="Treat none")], markers=False)
        fig.tight_layout(rect=(0, 0.12, 1, 1), h_pad=1.2, w_pad=1.0)
        save(fig, f"figure_dca_{tau}m.png", tif="Fig4.tif" if tau == 120 else None)
        plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, model in zip(axes, sm.CLASSIFIERS):
    g = metrics[metrics["model"] == model].groupby(["cell", "tau"])["brier"].mean().unstack("tau")
    for cell in CELL_ORDER:
        ax.plot(sm.HORIZONS, g.loc[cell, list(sm.HORIZONS)], marker="o", ls=STYLE[cell],
                color=COLOR[cell.split("+")[0]], label=cell)
    ax.set_title(LABELS[model])
    ax.set_xticks(sm.HORIZONS)
    ax.set_xlabel("Horizon (months)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("IPCW Brier score (mean over splits)")
axes[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(EVAL_DIR / "figure_divergence_brier.png", dpi=200)
plt.show()

## Tables S4–S5 — conventional 0.5 cut-off (supplement, illustrative)

This section uses the same test-fold predictions and IPCW weights as above, but classifies every model at the fixed software-default cut-off of 0.5 instead of its own γ\*. Nothing is refitted. It illustrates how the conventional analysis behaves and does not replace the prespecified threshold protocol.
- **Table S4:** all models and cells at 0.5.
- **Table S5:** the effect of SMOTE-NC in the Zero cells (common practice), at 0.5 and at the matched operating point (sensitivity ≥ 0.80, γ\*).

In [ ]:
CUTOFF = 0.5
CUT_METRICS = ["sensitivity", "specificity", "fpr", "precision", "f1"]
rows = []
for r, k in SPLITS:
    split = sm.load_split(MODEL_DIR, r, k)
    te = split["role"].eq("test").to_numpy()
    rows_te = split["row"].to_numpy()[te]
    for tau in sm.HORIZONS:
        y = data.labels[f"y_{tau}"].to_numpy()[rows_te]
        det = y >= 0
        yt, wt = (y[det] == 1).astype(int), split[f"w_{tau}"].to_numpy()[te][det]
        for col in [c for c in split.columns if c.endswith(f"|{tau}")]:
            model, cell, _ = col.split("|")
            p = split[col].to_numpy()[te][det].astype(float)
            rows.append({"repeat": r, "fold": k, "tau": tau, "model": model, "cell": cell,
                         **sm.classification_metrics(p, yt, wt, CUTOFF)})
cutoff = pd.DataFrame(rows)
cutoff.to_csv(EVAL_DIR / "metrics_cutoff_0.5_by_split.csv", index=False)

table_s4 = cutoff.groupby(["tau", "model", "cell"], sort=False)[CUT_METRICS].agg(lambda s: mean_sd(s, 3)).reset_index()
table_s4["model"] = pd.Categorical(table_s4["model"], MODEL_ORDER)
table_s4["cell"] = pd.Categorical(table_s4["cell"], CELL_ORDER + ["-"])
table_s4 = table_s4.sort_values(["tau", "model", "cell"])
table_s4.to_csv(EVAL_DIR / "table_S4_cutoff_0.5.csv", index=False)
display(table_s4.set_index(["tau", "model", "cell"]))

In [ ]:
rows = []
for tau in sm.HORIZONS:
    for model in sm.CLASSIFIERS:
        for cell in ["zero", "zero+smote"]:
            at_05 = cutoff[(cutoff["tau"] == tau) & (cutoff["model"] == model) & (cutoff["cell"] == cell)]
            at_gamma = metrics[(metrics["tau"] == tau) & (metrics["model"] == model) & (metrics["cell"] == cell)]
            rows.append({
                "tau": tau, "model": model, "smote": "yes" if "smote" in cell else "no",
                **{f"{m}_at_0.5": mean_sd(at_05[m], 3) for m in ["sensitivity", "fpr", "f1"]},
                **{f"{m}_at_gamma": mean_sd(at_gamma[m], 3) for m in ["sensitivity", "fpr", "f1"]},
                "gamma": mean_sd(at_gamma["gamma"], 3),
            })
table_s5 = pd.DataFrame(rows)
table_s5.to_csv(EVAL_DIR / "table_S5_smote_operating_point.csv", index=False)
display(table_s5.set_index(["tau", "model", "smote"]))

## Research-question summaries (for the Results and Discussion)

The four approaches in the research questions map onto cells as follows:
- **None** = Zero;
- **SMOTE-NC** = Zero + SMOTE-NC;
- **IPCW** = IPCW;
- **IPCW + SMOTE-NC** = IPCW + SMOTE-NC.

Discard is kept as an additional comparator.

The section computes three summaries:
- **Paired contrasts** between approaches for the Brier score, AUC, PPV at γ\* and O/E, with corrected 95% CIs (Methods Section 5.1).
- **PPV at exactly 80% sensitivity on each test fold.** This is a point on the precision–recall curve, used only to compare models at an identical sensitivity. It is not a decision rule; the operating thresholds remain γ\*.
- **Net benefit** at threshold probabilities of 5%, 10% and 20%.

In [ ]:
APPROACH = {"None": "zero", "Discard": "discard", "IPCW": "ipcw", "SMOTE-NC": "zero+smote", "IPCW + SMOTE-NC": "ipcw+smote"}
RQ_CONTRASTS = [("IPCW", "None"), ("IPCW", "Discard"), ("SMOTE-NC", "None"), ("IPCW + SMOTE-NC", "IPCW"), ("IPCW + SMOTE-NC", "None")]
rows = []
for (tau, model), g in grid.groupby(["tau", "model"]):
    for metric in ["brier", "auc", "precision", "oe_ratio"]:
        wide = g.pivot(index="split", columns="cell", values=metric)
        for a, b in RQ_CONTRASTS:
            res = sm.corrected_ttest((wide[APPROACH[a]] - wide[APPROACH[b]]).to_numpy())
            rows.append({"tau": tau, "model": model, "metric": metric, "contrast": f"{a} − {b}", **res})
table_rq = pd.DataFrame(rows)
table_rq.to_csv(EVAL_DIR / "table_RQ_contrasts.csv", index=False)
display(table_rq[table_rq["metric"] == "brier"].pivot_table(index=["tau", "contrast"], columns="model", values="mean").round(4))

In [ ]:
NB_THRESHOLDS = np.array([0.05, 0.10, 0.20])
rows = []
for r, k in SPLITS:
    split = sm.load_split(MODEL_DIR, r, k)
    te = split["role"].eq("test").to_numpy()
    rows_te = split["row"].to_numpy()[te]
    for tau in sm.HORIZONS:
        y = data.labels[f"y_{tau}"].to_numpy()[rows_te]
        det = y >= 0
        yt, wt = (y[det] == 1).astype(int), split[f"w_{tau}"].to_numpy()[te][det]
        for col in [c for c in split.columns if c.endswith(f"|{tau}")]:
            model, cell, _ = col.split("|")
            p = split[col].to_numpy()[te][det].astype(float)
            ppv80 = sm.classification_metrics(p, yt, wt, sm.select_threshold(p, yt, wt))["precision"]
            nb_vals = sm.net_benefit(p, yt, wt, int(te.sum()), NB_THRESHOLDS)
            rows.append({"tau": tau, "model": model, "cell": cell, "ppv_at_exact_80": ppv80,
                         **{f"net_benefit_{pt:.2f}": v for pt, v in zip(NB_THRESHOLDS, nb_vals)}})
rq_extra = pd.DataFrame(rows).groupby(["tau", "model", "cell"], sort=False).agg(["mean", "std"])
rq_extra.columns = [f"{a}_{b}" for a, b in rq_extra.columns]
rq_extra = rq_extra.reset_index()
rq_extra.to_csv(EVAL_DIR / "table_RQ_ppv80_netbenefit.csv", index=False)
display(rq_extra.round(4))

## Table S6 and Figure S1 — PPV at matched sensitivities (supplement, illustrative)

This section asks whether oversampling helps at any operating point, not only at sensitivity 0.80. For each test fold, classifier, horizon, censoring strategy and target sensitivity s ∈ {0.30, 0.40, …, 0.90, 0.95}:
1. The threshold is set to the highest cut-off reaching an IPCW-weighted sensitivity ≥ s, separately for the model without and the model with SMOTE-NC.
2. Their PPVs are compared.

The comparison reads both models' precision–recall curves at the same sensitivity. Thresholds are therefore set on the test fold itself and treated identically for both models; this compares the models, not a deployable rule. Differences are averaged over the 15 splits and tested with the corrected repeated-CV t-test.

In [ ]:
SENS_GRID = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
rows = []
for r, k in SPLITS:
    split = sm.load_split(MODEL_DIR, r, k)
    te = split["role"].eq("test").to_numpy()
    rows_te = split["row"].to_numpy()[te]
    for tau in sm.HORIZONS:
        y = data.labels[f"y_{tau}"].to_numpy()[rows_te]
        det = y >= 0
        yt, wt = (y[det] == 1).astype(int), split[f"w_{tau}"].to_numpy()[te][det]
        for model in sm.CLASSIFIERS:
            for rule in sm.CENSORING_RULES:
                for smote in (False, True):
                    p = split[sm.pred_column(model, sm.cell_name(rule, smote), tau)].to_numpy()[te][det].astype(float)
                    for target in SENS_GRID:
                        gamma = sm.select_threshold(p, yt, wt, target=target)
                        rows.append({"repeat": r, "fold": k, "tau": tau, "model": model, "rule": rule,
                                     "smote": smote, "sensitivity_target": target,
                                     "ppv": sm.classification_metrics(p, yt, wt, gamma)["precision"]})
matched = pd.DataFrame(rows)
matched.to_csv(EVAL_DIR / "metrics_ppv_matched_sensitivity_by_split.csv", index=False)

paired = matched.pivot_table(index=["repeat", "fold", "tau", "model", "rule", "sensitivity_target"],
                             columns="smote", values="ppv").reset_index()
paired["diff"] = paired[True] - paired[False]
rows = []
for (tau, model, rule, target), g in paired.groupby(["tau", "model", "rule", "sensitivity_target"]):
    rows.append({"tau": tau, "model": model, "rule": rule, "sensitivity_target": target,
                 "ppv_no_smote": g[False].mean(), "ppv_smote": g[True].mean(),
                 **sm.corrected_ttest(g["diff"].to_numpy()), "splits_smote_higher": int((g["diff"] > 0).sum())})
table_s6 = pd.DataFrame(rows)
table_s6["p_holm"] = sm.holm_adjust(table_s6["p_value"].to_numpy())
print(f"Significant after Holm correction ({len(table_s6)} comparisons): {(table_s6['p_holm'] < 0.05).sum()}")
table_s6.to_csv(EVAL_DIR / "table_S6_ppv_matched_sensitivity.csv", index=False)
print(f"Comparisons: {len(table_s6)} | SMOTE-NC PPV >= no oversampling on average: {(table_s6['mean'] >= 0).sum()} | "
      f"significantly lower (p < 0.05): {((table_s6['mean'] < 0) & (table_s6['p_value'] < 0.05)).sum()} | "
      f"split-level SMOTE-NC wins: {table_s6['splits_smote_higher'].sum()} of {len(paired)}")
display(table_s6.pivot_table(index=["tau", "model", "rule"], columns="sensitivity_target", values="mean").round(3))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10), sharex=True)
for i, tau in enumerate(sm.HORIZONS):
    for j, model in enumerate(sm.CLASSIFIERS):
        ax = axes[i, j]
        for rule in sm.CENSORING_RULES:
            g = table_s6[(table_s6["tau"] == tau) & (table_s6["model"] == model) & (table_s6["rule"] == rule)]
            ax.plot(g["sensitivity_target"], g["mean"], marker="o", ms=3, color=COLOR[rule],
                    label={"zero": "Zero", "discard": "Discard", "ipcw": "IPCW"}[rule])
            ax.fill_between(g["sensitivity_target"], g["ci_low"], g["ci_high"], color=COLOR[rule], alpha=0.15)
        ax.axhline(0, color="black", lw=0.8)
        ax.set_title(f"{LABELS[model]}, {tau} months", fontsize=10)
        ax.grid(alpha=0.3)
        if j == 0:
            ax.set_ylabel("PPV(SMOTE-NC) − PPV(no oversampling)")
        if i == 2:
            ax.set_xlabel("Matched sensitivity")
axes[0, 0].legend(title="Censoring strategy", fontsize=8)
plt.tight_layout()
fig.savefig(EVAL_DIR / "figure_S1_ppv_matched_sensitivity.png", dpi=200)
plt.show()

## Sensitivity checks requested in the v9 review (`Documents/Review_v9_assessment.md`, M1 and C3)

- **M1, weight truncation:** test-fold Brier scores with and without the 99th-percentile truncation of the IPCW weights. The censoring model of each split is refitted exactly as in `03_modeling` (`sm.test_weights_untruncated`); no classifier is refitted, and the refitted weights are checked against the saved ones.
- **C3, observed risk:** O/E with the Aalen–Johansen observed risk (used in the paper) and with the IPCW-weighted observed risk, and the treat-all net benefit under both estimators.

In [ ]:
rows = []
for r, k in SPLITS:
    split = sm.load_split(MODEL_DIR, r, k)
    te = split["role"].eq("test").to_numpy()
    rows_te = split["row"].to_numpy()[te]
    n_test = int(te.sum())
    w_raw_all = sm.test_weights_untruncated(data, r, k)
    for tau in sm.HORIZONS:
        y = data.labels[f"y_{tau}"].to_numpy()[rows_te]
        det = y >= 0
        yt = (y[det] == 1).astype(int)
        w_tr, w_raw = split[f"w_{tau}"].to_numpy()[te].astype(float), w_raw_all[tau]
        assert np.allclose(np.minimum(w_raw, w_tr.max()), w_tr, atol=1e-5), "refitted weights differ from saved"
        risk_ipcw = np.sum(w_tr[det] * yt) / np.sum(w_tr[det])
        risk_aj = sm.aalen_johansen_cif1(V_ALL[rows_te], S_ALL[rows_te], tau)
        for col in [c for c in split.columns if c.endswith(f"|{tau}")]:
            model, cell, _ = col.split("|")
            p = split[col].to_numpy()[te].astype(float)
            rows.append({"repeat": r, "fold": k, "tau": tau, "model": model, "cell": cell,
                         "brier_truncated": np.sum(w_tr[det] * (yt - p[det]) ** 2) / n_test,
                         "brier_untruncated": np.sum(w_raw[det] * (yt - p[det]) ** 2) / n_test,
                         "oe_aj": risk_aj / p.mean(), "oe_ipcw": risk_ipcw / p.mean(),
                         "risk_aj": risk_aj, "risk_ipcw": risk_ipcw})
sens = pd.DataFrame(rows).assign(split=lambda d: list(zip(d["repeat"], d["fold"])))
sens.drop(columns="split").to_csv(EVAL_DIR / "metrics_sensitivity_truncation_oe_by_split.csv", index=False)

rows = []   # untruncated Brier: same contrasts as Table 7
for (tau, model), g in sens[sens["model"].isin(sm.CLASSIFIERS)].groupby(["tau", "model"]):
    wide = g.pivot(index="split", columns="cell", values="brier_untruncated")
    for a, b in RQ_CONTRASTS:
        rows.append({"tau": tau, "model": model, "contrast": f"{a} − {b}",
                     **sm.corrected_ttest((wide[APPROACH[a]] - wide[APPROACH[b]]).to_numpy())})
pd.DataFrame(rows).to_csv(EVAL_DIR / "table_sensitivity_untruncated_brier.csv", index=False)

summary = sens.groupby(["tau", "model", "cell"])[["brier_truncated", "brier_untruncated", "oe_aj", "oe_ipcw"]].mean()
summary.to_csv(EVAL_DIR / "table_sensitivity_truncation_oe.csv")
rank = lambda col: summary[col].groupby(["tau", "model"]).rank()
print("Cell ranking unchanged without truncation:", bool((rank("brier_truncated") == rank("brier_untruncated")).all()))
print("O/E difference, IPCW minus Aalen-Johansen, by horizon:")
display((summary["oe_ipcw"] - summary["oe_aj"]).groupby("tau").agg(["mean", "min", "max"]).round(4))
risk = sens.drop_duplicates(["repeat", "fold", "tau"]).groupby("tau")[["risk_aj", "risk_ipcw"]].mean()
for pt in NB_THRESHOLDS:
    for est in ("aj", "ipcw"):
        risk[f"treat_all_{est}_{pt:.2f}"] = risk[f"risk_{est}"] - (1 - risk[f"risk_{est}"]) * pt / (1 - pt)
display(risk.round(4))